# Fast Tokenizers in the QA Pipeline

**Source:** [HuggingFace LLM Course – Chapter 6, Section 3b](https://huggingface.co/learn/llm-course/chapter6/3b)

In Question Answering (QA), the model is tasked with finding a substring (span) within a given context that answers a question. In this notebook, we explore how fast tokenizers handle complex QA logic, particularly the challenge of **long contexts** that exceed the model's maximum sequence length.

We will manually reproduce the logic of the QA `pipeline`, utilizing fast tokenizers' **offset mapping** and **sliding windows (stride)** to successfully extract answers from massive documents.

---

## Step 1: Initialize QA Pipeline

We begin by loading the default Question Answering pipeline from the Hugging Face `transformers` library.

In [22]:
from transformers import pipeline

question_answerer = pipeline("question-answering")

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Step 2: Test Pipeline on a Short Context

Let's test the pipeline on a standard, short context to see how it extracts the exact answer from the provided text.

In [23]:
context = """
🤗 Transformers is backed by the three most popular deep learning libraries — Jax, PyTorch, and TensorFlow — with a seamless integration
between them. It's straightforward to train your models with one before loading them for inference with the other.
"""
question = "Which deep learning libraries back 🤗 Transformers?"
question_answerer(question=question, context=context)

{'score': 0.9804228239154327,
 'start': 78,
 'end': 106,
 'answer': 'Jax, PyTorch, and TensorFlow'}

## Step 3: Test Pipeline on a Long Context

Next, we pass a much longer context. Note that the pipeline still effortlessly finds the correct answer hidden deep within the text. How does it handle contexts that exceed the model's maximum sequence length? We will explore that shortly.

In [24]:
long_context = """
🤗 Transformers: State of the Art NLP

🤗 Transformers provides thousands of pretrained models to perform tasks on texts such as classification, information extraction,
question answering, summarization, translation, text generation and more in over 100 languages.
Its aim is to make cutting-edge NLP easier to use for everyone.

🤗 Transformers provides APIs to quickly download and use those pretrained models on a given text, fine-tune them on your own datasets and
then share them with the community on our model hub. At the same time, each python module defining an architecture is fully standalone and
can be modified to enable quick research experiments.

Why should I use transformers?

1. Easy-to-use state-of-the-art models:
  - High performance on NLU and NLG tasks.
  - Low barrier to entry for educators and practitioners.
  - Few user-facing abstractions with just three classes to learn.
  - A unified API for using all our pretrained models.
  - Lower compute costs, smaller carbon footprint:

2. Researchers can share trained models instead of always retraining.
  - Practitioners can reduce compute time and production costs.
  - Dozens of architectures with over 10,000 pretrained models, some in more than 100 languages.

3. Choose the right framework for every part of a model's lifetime:
  - Train state-of-the-art models in 3 lines of code.
  - Move a single model between TF2.0/PyTorch frameworks at will.
  - Seamlessly pick the right framework for training, evaluation and production.

4. Easily customize a model or an example to your needs:
  - We provide examples for each architecture to reproduce the results published by its original authors.
  - Model internals are exposed as consistently as possible.
  - Model files can be used independently of the library for quick experiments.

🤗 Transformers is backed by the three most popular deep learning libraries — Jax, PyTorch and TensorFlow — with a seamless integration
between them. It's straightforward to train your models with one before loading them for inference with the other.
"""
question_answerer(question=question, context=long_context)

{'score': 0.9717117228428833,
 'start': 1892,
 'end': 1919,
 'answer': 'Jax, PyTorch and TensorFlow'}

## Step 4: Manual QA Processing (Without Pipeline)

To understand what happens under the hood, let's load the model and tokenizer manually. We tokenize the question and context together.

In [25]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

model_checkpoint = "distilbert-base-cased-distilled-squad"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

inputs = tokenizer(question, context, return_tensors="pt")
outputs = model(**inputs)

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

## Step 5: Extract Logits

The model outputs two tensors: `start_logits` and `end_logits`. Each has a shape of `(1, sequence_length)` representing the probability of each token being the start or end of the answer.

In [26]:
start_logits = outputs.start_logits
end_logits = outputs.end_logits
print(start_logits.shape, end_logits.shape)

torch.Size([1, 67]) torch.Size([1, 67])


## Step 6: Masking Invalid Tokens

We cannot have the answer start or end in the *question* part of the sequence, nor on special tokens like `[SEP]`. We use `sequence_ids()` to identify which tokens belong to the context (sequence ID `1`), and mask the others with a large negative number so their probabilities become zero.

In [27]:
import torch

sequence_ids = inputs.sequence_ids()
# Mask everything apart from the tokens of the context
mask = [i != 1 for i in sequence_ids]
# Unmask the [CLS] token
mask[0] = False
mask = torch.tensor(mask)[None]

start_logits[mask] = -10000
end_logits[mask] = -10000

## Step 7: Calculate Probabilities

We apply the softmax function to convert the raw logits into proper probabilities for the start and end positions.

In [28]:
start_probabilities = torch.nn.functional.softmax(start_logits, dim=-1)[0]
end_probabilities = torch.nn.functional.softmax(end_logits, dim=-1)[0]

## Step 8: Calculate All Possible Span Scores

We compute the score for every possible answer span by multiplying the probability of each start position with the probability of each end position. This gives us a 2D matrix of scores.

In [29]:
scores = start_probabilities[:, None] * end_probabilities[None, :]

## Step 9: Mask Invalid Spans (End before Start)

An answer cannot end before it starts! We use `torch.triu()` to zero out the lower triangle of our score matrix, ensuring `start_index <= end_index`.

In [30]:
scores = torch.triu(scores)

## Step 10: Find the Best Span

We find the index of the highest score in our matrix and convert it back to the specific `start_index` and `end_index` for the answer span.

In [31]:
max_index = scores.argmax().item()
start_index = max_index // scores.shape[1]
end_index = max_index % scores.shape[1]
print(scores[start_index, end_index])

tensor(0.9803, grad_fn=<SelectBackward0>)


## Step 11: Map Tokens to Characters

Using the fast tokenizer's `return_offsets_mapping=True`, we map our predicted token indices back to the exact character spans in the original context string.

In [32]:
inputs_with_offsets = tokenizer(question, context, return_offsets_mapping=True)
offsets = inputs_with_offsets["offset_mapping"]

start_char, _ = offsets[start_index]
_, end_char = offsets[end_index]
answer = context[start_char:end_char]

## Step 12: Final Output

We package the extracted answer substring, its character offsets, and the confidence score. This matches the output format of the high-level `pipeline`.

In [33]:
result = {
    "answer": answer,
    "start": start_char,
    "end": end_char,
    "score": scores[start_index, end_index],
}
print(result)

{'answer': 'Jax, PyTorch, and TensorFlow', 'start': 78, 'end': 106, 'score': tensor(0.9803, grad_fn=<SelectBackward0>)}


## Step 13: The Long Context Problem

If we tokenize our long context example, it exceeds the model's maximum length (typically 384 tokens). A standard tokenizer would just truncate the end, potentially throwing away the actual answer.

In [34]:
inputs = tokenizer(question, long_context)
print(len(inputs["input_ids"]))

461


## Step 14: Standard Truncation

If we force truncation on the context (`only_second`), we can see from the decoded text that the sentence containing the answer was completely chopped off!

In [35]:
inputs = tokenizer(question, long_context, max_length=384, truncation="only_second")
print(tokenizer.decode(inputs["input_ids"]))

[CLS] Which deep learning libraries back [UNK] Transformers? [SEP] [UNK] Transformers : State of the Art NLP [UNK] Transformers provides thousands of pretrained models to perform tasks on texts such as classification, information extraction, question answering, summarization, translation, text generation and more in over 100 languages. Its aim is to make cutting - edge NLP easier to use for everyone. [UNK] Transformers provides APIs to quickly download and use those pretrained models on a given text, fine - tune them on your own datasets and then share them with the community on our model hub. At the same time, each python module defining an architecture is fully standalone and can be modified to enable quick research experiments. Why should I use transformers? 1. Easy - to - use state - of - the - art models : - High performance on NLU and NLG tasks. - Low barrier to entry for educators and practitioners. - Few user - facing abstractions with just three classes to learn. - A unified A

## Step 15: Sliding Windows with Stride

To solve this, fast tokenizers support a sliding window approach. By setting `return_overflowing_tokens=True` and specifying a `stride`, the tokenizer splits the long context into multiple overlapping chunks (features). The stride ensures that an answer split across a boundary might still be captured whole in the adjacent chunk.

In [36]:
sentence = "This sentence is not too long but we are going to split it anyway."
inputs = tokenizer(
    sentence, truncation=True, return_overflowing_tokens=True, max_length=6, stride=2
)

for ids in inputs["input_ids"]:
    print(tokenizer.decode(ids))

[CLS] This sentence is not [SEP]
[CLS] is not too long [SEP]
[CLS] too long but we [SEP]
[CLS] but we are going [SEP]
[CLS] are going to split [SEP]
[CLS] to split it anyway [SEP]
[CLS] it anyway. [SEP]


## Step 16: Inspect Overflowing Tokens

The tokenizer returns a batch of features. Notice the new `overflow_to_sample_mapping` key.

In [37]:
print(inputs.keys())

KeysView({'input_ids': [[101, 1188, 5650, 1110, 1136, 102], [101, 1110, 1136, 1315, 1263, 102], [101, 1315, 1263, 1133, 1195, 102], [101, 1133, 1195, 1132, 1280, 102], [101, 1132, 1280, 1106, 3325, 102], [101, 1106, 3325, 1122, 4050, 102], [101, 1122, 4050, 119, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1]], 'overflow_to_sample_mapping': [0, 0, 0, 0, 0, 0, 0]})


## Step 17: Overflow to Sample Mapping

This mapping tells us which original string each generated feature (chunk) belongs to. Here, all chunks map back to sample `0` (our single input sentence).

In [39]:
print(inputs["overflow_to_sample_mapping"])

[0, 0, 0, 0, 0, 0, 0]


## Step 15: Sliding Windows with Stride

To solve this, fast tokenizers support a sliding window approach. By setting `return_overflowing_tokens=True` and specifying a `stride`, the tokenizer splits the long context into multiple overlapping chunks (features). The stride ensures that an answer split across a boundary might still be captured whole in the adjacent chunk.

In [40]:
sentences = [
    "This sentence is not too long but we are going to split it anyway.",
    "This sentence is shorter but will still get split.",
]
inputs = tokenizer(
    sentences, truncation=True, return_overflowing_tokens=True, max_length=6, stride=2
)

print(inputs["overflow_to_sample_mapping"])

[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1]


## Step 19: Tokenize Long Context with Stride

Now let's apply this to our actual long QA context. We use a max length of 384 and a stride of 128 overlapping tokens.

In [41]:
inputs = tokenizer(
    question,
    long_context,
    stride=128,
    max_length=384,
    padding="longest",
    truncation="only_second",
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
)

## Step 20: Prepare Tensors for the Model

We remove the mapping keys (since the model doesn't expect them) and convert our input features into PyTorch tensors. We now have a batch size of 2 (two chunks for the one long context).

In [42]:
_ = inputs.pop("overflow_to_sample_mapping")
offsets = inputs.pop("offset_mapping")

inputs = inputs.convert_to_tensors("pt")
print(inputs["input_ids"].shape)

torch.Size([2, 384])


## Step 21: Model Prediction on Chunks

We pass both chunks through the model simultaneously. The output shapes are now `(2, 384)`.

In [43]:
outputs = model(**inputs)

start_logits = outputs.start_logits
end_logits = outputs.end_logits
print(start_logits.shape, end_logits.shape)

torch.Size([2, 384]) torch.Size([2, 384])


## Step 22: Advanced Masking for Padding

Because our chunks might have padding, we must also mask out the `[PAD]` tokens in addition to masking the question tokens, so the model doesn't predict an answer in the padding.

In [44]:
sequence_ids = inputs.sequence_ids()
# Mask everything apart from the tokens of the context
mask = [i != 1 for i in sequence_ids]
# Unmask the [CLS] token
mask[0] = False
# Mask all the [PAD] tokens
mask = torch.logical_or(torch.tensor(mask)[None], (inputs["attention_mask"] == 0))

start_logits[mask] = -10000
end_logits[mask] = -10000

## Step 23: Batch Probabilities

We calculate the softmax probabilities for both chunks.

In [45]:
start_probabilities = torch.nn.functional.softmax(start_logits, dim=-1)
end_probabilities = torch.nn.functional.softmax(end_logits, dim=-1)

## Step 24: Find the Best Candidate per Chunk

We iterate through our chunks, calculate the valid score matrices (masking the lower triangle), and extract the best predicted answer candidate for *each* chunk.

In [46]:
candidates = []
for start_probs, end_probs in zip(start_probabilities, end_probabilities):
    scores = start_probs[:, None] * end_probs[None, :]
    idx = torch.triu(scores).argmax().item()

    start_idx = idx // scores.shape[1]
    end_idx = idx % scores.shape[1]
    score = scores[start_idx, end_idx].item()
    candidates.append((start_idx, end_idx, score))

print(candidates)

[(0, 18, 0.3386707007884979), (173, 184, 0.9714869856834412)]


## Step 25: Compare Candidates

Finally, we map the candidates back to characters and compare their scores. The chunk that actually contained the answer has a vastly higher confidence score (`0.97`) than the chunk that didn't (`0.33`), allowing us to confidently select the final answer!

In [47]:
for candidate, offset in zip(candidates, offsets):
    start_token, end_token, score = candidate
    start_char, _ = offset[start_token]
    _, end_char = offset[end_token]
    answer = long_context[start_char:end_char]
    result = {"answer": answer, "start": start_char, "end": end_char, "score": score}
    print(result)

{'answer': '\n🤗 Transformers: State of the Art NLP', 'start': 0, 'end': 37, 'score': 0.3386707007884979}
{'answer': 'Jax, PyTorch and TensorFlow', 'start': 1892, 'end': 1919, 'score': 0.9714869856834412}
